# CASMI26 - fingerprint networks, FULL-data retrain seed 30

Trains the per-spectrum and the merged spectrum->fingerprint networks on a T4 and writes
`/kaggle/working/out/export/` (`fp_single_fulls30.pt`, `fp_merged_fulls30.pt`, `train_result.json`, logs).

Settings: **Accelerator = GPU T4 x2 (or T4)**, Internet off is fine, private dataset attached.
Use **Save Version -> Save & Run All (Commit)**.

In [ ]:
import glob, os, subprocess, sys
hits = sorted(glob.glob('/kaggle/input/**/train_fp.py', recursive=True))
assert hits, 'attach the training dataset (casmi-train-pkg-full)'
PKG = os.path.dirname(hits[0]); print('package:', PKG)
prev = sorted(set(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/ckpt_*.pt', recursive=True)))
print('earlier checkpoints:', prev or 'none')
!nvidia-smi -L
import torch; print(torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'turn on the GPU: Settings -> Accelerator -> GPU T4'

In [ ]:
# Seed 30 run
SEED_OFFSET, TAG = 30, 'fulls30'
s = open(f'{PKG}/train_fp.py').read()
if '--seed_offset' not in s:
    old = '    a = ap.parse_args()\n'
    assert s.count(old) == 1
    s = s.replace(old, old + '    for _n, _s in MODELS.items():\n        _s["seed"] += SEED_OFFSET_; _s["file"] = f"fp_{_n}_{TAG_}.pt"\n')
    s = s.replace('SEED_OFFSET_', str(SEED_OFFSET)).replace('TAG_', repr(TAG))
import shutil
for f in os.listdir(PKG):
    if f.endswith('.py'): shutil.copy(f'{PKG}/{f}', f'/kaggle/working/{f}')
open('/kaggle/working/train_fp.py', 'w').write(s)
cmd = [sys.executable, '/kaggle/working/train_fp.py', '--data', PKG, '--out', '/kaggle/working/out', '--max_minutes', '660']
if prev: cmd += ['--resume_from', prev[0]]
rc = subprocess.call(cmd); print('exit code', rc, '(0 = finished, 2 = not finished yet, run again)')

In [ ]:
rc2 = subprocess.call([sys.executable, '/kaggle/working/check_result.py', '--data', PKG, '--out', '/kaggle/working/out'])
print(open('/kaggle/working/out/export/check_summary.txt').read())
!ls -la /kaggle/working/out /kaggle/working/out/export